# Capability Lab — Private Validation Bootstrap

Public bootstrap sem credenciais ou código privado. Ele autentica via GitHub CLI, clona o repositório privado e valida o snapshot exato do Team Board ADK.

**Não cole tokens neste notebook.** Use apenas o fluxo de autenticação do GitHub aberto pela célula.


In [ ]:
import subprocess, shutil, pathlib, os, sys

def run(*cmd, check=True):
    print('\n$', ' '.join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), check=check, text=True)

if shutil.which('gh') is None:
    run('bash','-lc','apt-get update -qq && apt-get install -y -qq gh')

print('gh:', shutil.which('gh'))


In [ ]:
status = subprocess.run(['gh','auth','status','-h','github.com'])
if status.returncode != 0:
    run('gh','auth','login','--hostname','github.com','--git-protocol','https','--web')
run('gh','auth','setup-git')
run('gh','auth','status','-h','github.com')


In [ ]:
ROOT = pathlib.Path('/content/caplab-team-board')
if ROOT.exists():
    run('rm','-rf',ROOT)
run('gh','repo','clone','lucas-mateus-hq/lucas-capability-os',ROOT,'--','--filter=blob:none','--no-checkout')
run('git','-C',ROOT,'fetch','--depth','1','origin','a0da6ef9a33913a52fec22c6334d89a0374e2901')
run('git','-C',ROOT,'checkout','--detach','a0da6ef9a33913a52fec22c6334d89a0374e2901')
os.chdir(ROOT)
print('\n=== EXACT HEAD ===')
run('git','rev-parse','HEAD')
run('git','status','--porcelain')


In [ ]:
print('\n=== STATIC TEAM GATE ===')
run(sys.executable,'team/google_native/scripts/static_gate.py')

print('\n=== TEAM BOARD CONTRACT ===')
run(sys.executable,'-m','pytest','-q','tests/test_team_board_contract.py')

print('\n=== INSTALL GOOGLE-NATIVE DEPS ===')
run(sys.executable,'-m','pip','install','-q','-r','team/google_native/requirements.txt')

print('\n=== ADK IMPORT ===')
run(sys.executable,'-c',"from team.google_native.app.agent import root_agent; print('ADK_IMPORT_PASS', root_agent.name)")

print('\n=== SOURCE MANIFEST VERIFY ===')
manifest = subprocess.run([sys.executable,'scripts/verify_source_manifest.py'], text=True)

print('\n=== FINAL ===')
print('SOURCE_MANIFEST_EXIT =', manifest.returncode)
print('HEAD =', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
